# 3D toric code — QMC observables vs field, L=4..12

Pure-QMC (ParaToric) validity check across both cross-validation arcs — the electric line
($h_x=0.2$, sweep $h_z$) and the magnetic line ($h_z=0.1$, sweep $h_x$) — now extended past
the NQS-matched $L\in\{4,5,6\}$ range to $L\in\{8,10,12\}$ (near-critical windows,
2026-08-15 campaign). Every point carries the full standard observable set (energy, both
stabilizers, both magnetizations) plus the relevant topological order-parameter ratio
(Z-string on the electric arc, both membrane families on the magnetic arc). No NQS data —
this is a QMC-only sanity/trend check; see `tune_rect_summary.ipynb` /
`vertical_line_hz.ipynb` / `xz_cut.ipynb` for NQS-vs-QMC comparisons.

## 1 · Config

In [ ]:
# ====================== 1 · CONFIG — the one cell to edit ======================
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})

ROOT = (os.path.abspath(os.path.join(os.getcwd(), "..", "..", "results"))
        if os.getcwd().endswith(os.path.join("analysis", "notebooks")) else "results")
FIGDIR = os.path.join(os.path.dirname(ROOT), "figures")  # repo-root figures/ (gitignored paper target), cwd-safe like ROOT
os.makedirs(FIGDIR, exist_ok=True)

# arc definitions: (directory glob, canonical basis tag, [(json_key, short_name), ...])
ARCS = {
    "electric": dict(dirglob=f"{ROOT}/qmc_hx0.2_hz*", basis_tag="bz", field_key="hz",
                      fm_keys=[("fredenhagen_marcu", "FM")], title="Electric arc ($h_x=0.2$)"),
    "magnetic": dict(dirglob=f"{ROOT}/qmc_hx*_hz0.1", basis_tag="bx", field_key="hx",
                      fm_keys=[("fredenhagen_marcu_membrane_r1", "FMr1"),
                               ("fredenhagen_marcu_membrane", "FMmem")],
                      title="Magnetic arc ($h_z=0.1$)"),
}

# known exact transitions (reference lines only, not fit inputs)
HZ_C = 0.193869   # electric, continuous
HX_C = 1.0        # magnetic, first-order

ALL_LS = [4, 5, 6, 7, 8, 10, 12]
COL = dict(zip(ALL_LS, plt.cm.plasma(np.linspace(0, 0.85, len(ALL_LS)))))


## 2 · Data library

Loads every `paratoric_L*.json` under each arc's field-point directories. When multiple runs
exist for the same $(L, h_x, h_z)$ (older multi-$\beta$ sweeps, `_combined.json` files),
prefers the `_combined.json` if present, else the lowest-energy-SEM file — never averages
across mismatched recipes.

In [ ]:
# ====================== 2 · DATA LIBRARY ======================
def n_edges(L):
    return 3 * L**2 * (L - 1)   # OBC cubic lattice edge count


def pick_best(files):
    combined = [f for f in files if "combined" in f]
    if combined:
        files = combined
    best, best_err = None, np.inf
    for f in files:
        d = json.load(open(f))
        err = d["combined"].get("energy", {}).get("sem", np.inf)
        if err is not None and err < best_err:
            best, best_err = d, err
    return best


def load_arc(dirglob, basis_tag, fm_keys):
    rows = []
    for d in sorted(glob.glob(dirglob)):
        for L in ALL_LS:
            files = glob.glob(f"{d}/paratoric_L{L}_*{basis_tag}*.json")
            if not files:
                continue
            rec = pick_best(files)
            if rec is None:
                continue
            c = rec["combined"]
            row = dict(L=L, hx=rec["hx"], hz=rec["hz"],
                       E=c.get("energy", {}).get("mean"), E_err=c.get("energy", {}).get("sem"),
                       chi2=c.get("energy", {}).get("chi2_red"),
                       Av=c.get("star_x", {}).get("mean"), Av_err=c.get("star_x", {}).get("sem"),
                       Bp=c.get("plaquette_z", {}).get("mean"), Bp_err=c.get("plaquette_z", {}).get("sem"),
                       Mx=c.get("sigma_x", {}).get("mean"), Mx_err=c.get("sigma_x", {}).get("sem"),
                       Mz=c.get("sigma_z", {}).get("mean"), Mz_err=c.get("sigma_z", {}).get("sem"))
            for jkey, short in fm_keys:
                fm = c.get(jkey, {})
                row[short] = fm.get("pooled", fm.get("mean"))
                row[short + "_err"] = fm.get("pooled_err", fm.get("sem"))
                row[short + "_denz"] = fm.get("den_z")
            rows.append(row)
    return rows


DATA = {name: load_arc(cfg["dirglob"], cfg["basis_tag"], cfg["fm_keys"]) for name, cfg in ARCS.items()}
for name, rows in DATA.items():
    Ls_present = sorted(set(r["L"] for r in rows))
    print(f"{name}: {len(rows)} points, L present = {Ls_present}")


## 3 · Curves vs field, both arcs

In [ ]:
# ====================== 3 · CURVES vs field ======================
def plot_arc(name):
    cfg = ARCS[name]
    rows, field_key = DATA[name], cfg["field_key"]
    Ls = sorted(set(r["L"] for r in rows))
    fm_short = cfg["fm_keys"][0][1]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    panels = [("E_per_edge", "E / N_edges"), ("Av", r"$\langle A_v\rangle$"),
              ("Bp", r"$\langle B_p\rangle$"), ("Mx", r"$\langle\sigma_x\rangle$"),
              ("Mz", r"$\langle\sigma_z\rangle$"), (fm_short, f"O_FM ({fm_short})")]
    for ax, (key, ylabel) in zip(axes.flat, panels):
        for L in Ls:
            pts = sorted([r for r in rows if r["L"] == L], key=lambda r: r[field_key])
            if not pts:
                continue
            x = [r[field_key] for r in pts]
            if key == "E_per_edge":
                y = [r["E"] / n_edges(r["L"]) if r["E"] is not None else np.nan for r in pts]
                yerr = [r["E_err"] / n_edges(r["L"]) if r["E_err"] is not None else 0 for r in pts]
            else:
                y = [r.get(key) for r in pts]
                yerr = [r.get(key + "_err") or 0 for r in pts]
            y = [np.nan if v is None else v for v in y]
            ax.errorbar(x, y, yerr=yerr, fmt="o-", ms=4, lw=1.2, color=COL[L], label=f"L={L}",
                        ecolor="0.5", capsize=2)
        if field_key == "hz":
            ax.axvline(HZ_C, color="k", ls=":", lw=1, zorder=0)
        else:
            ax.axvline(HX_C, color="k", ls=":", lw=1, zorder=0)
        ax.set(xlabel=field_key, ylabel=ylabel)
    axes.flat[0].legend(loc="upper left", fontsize=8)
    fig.suptitle(f"{cfg['title']}: observables vs ${field_key}$")
    fig.tight_layout()
    # fig.savefig(f"{FIGDIR}/qmc_{name}_arc_observables.png", dpi=300, bbox_inches="tight")
    plt.show()


for name in ARCS:
    plot_arc(name)


## 4 · Statistical health check

Flags any point whose energy $\chi^2_{\mathrm{red}}$ across chains/blocks is far from 1 —
the cheap tell for under- or over-estimated error bars from insufficient decorrelation
(see CLAUDE.md: "under-decorrelated runs finish cleanly and return biased energies with
confident error bars").

In [ ]:
# ====================== 4 · chi2_red SANITY ======================
print("chi2_red outliers (|chi2_red - 1| > 2):")
n_flagged = 0
for name, rows in DATA.items():
    for r in sorted(rows, key=lambda r: (r["L"], r[ARCS[name]["field_key"]])):
        if r["chi2"] is not None and abs(r["chi2"] - 1) > 2:
            print(f"  {name}: L={r['L']} hx={r['hx']} hz={r['hz']} chi2_red={r['chi2']:.2f}")
            n_flagged += 1
print(f"{n_flagged} flagged out of {sum(len(r) for r in DATA.values())} points" if n_flagged else
      f"none flagged, out of {sum(len(r) for r in DATA.values())} points — clean")
